In [ ]:
import os, glob, json, time
import numpy as np, pandas as pd
from sklearn.mixture import GaussianMixture
from sklearn.preprocessing import StandardScaler

BASE = r"E:\Abroad period research\Time-Series KD + Explainability\Multimodel codes"
STATIONS = {
    "Asomadilla":      ("Asomadilla Results",     "Asomadilla_2015_2023.ods"),
    "Aljarafe":        ("Aljarafe Results",       "Aljarafe.ods"),
    "Bermejales":      ("Bermejales Results",     "Bermejales_2015_2023.ods"),
    "Ronda del Valle": ("RonsadelVAalle Results", "RonsadelVAalle_2015_2023.ods"),
    "Torneo":          ("Torneo Results",         "Torneo_2015_2023.ods"),
}
TARGET_COL, DATETIME_COL, LAGS, SEED = "O3", "FECHA_HORA", [1,2,3,24], 42

def read_table(p): return pd.read_excel(p, engine="odf")
def normalize(df):
    df = df.copy(); df.columns = [str(c).strip() for c in df.columns]; r = {}
    for c in df.columns:
        cu = c.upper()
        if cu.startswith("DD") or cu in ("WD","DIR"):                 r[c]="DD"
        elif cu in ("VV","WS") or cu.startswith("VV") or cu.startswith("VEL"): r[c]="VV"
        elif "NO2" in cu:                                              r[c]="NO2"
        elif cu=="O3" or cu.endswith("-O3") or cu.startswith("O3"):   r[c]="O3"
        elif "PM10" in cu:                                             r[c]="PM10"
        elif "TMP" in cu or "TEMP" in cu:                              r[c]="TMP"
        elif "CO" in cu and "CO2" not in cu:                           r[c]="CO"
    return df.rename(columns=r)
def add_time(df, dt):
    df['hour']=df[dt].dt.hour; df['month']=df[dt].dt.month
    df['hour_sin']=np.sin(2*np.pi*df['hour']/24); df['hour_cos']=np.cos(2*np.pi*df['hour']/24)
    df['month_sin']=np.sin(2*np.pi*(df['month']-1)/12); df['month_cos']=np.cos(2*np.pi*(df['month']-1)/12)
    return df
def make_lags(df, cols, lags):
    for c in cols:
        for l in lags: df[f"{c}_lag{l}"] = df[c].shift(l)
    return df

def find_json(folder):
    for f in ("metrics_log.json", "comprehensive_dmoff_report.json"):
        hits = glob.glob(os.path.join(BASE, folder, "**", f), recursive=True)
        if hits: return hits[0]
    return None

def extract_model_time(js):
    d = json.load(open(js))
    t = d.get("train_times_s") or {}
    return sum(float(v) for v in t.values() if isinstance(v, (int, float)))

def fast_stage(path):
    T = {}
    t0=time.perf_counter(); df=normalize(read_table(path)); T['load']=time.perf_counter()-t0
    dt = DATETIME_COL if DATETIME_COL in df.columns else next(c for c in df.columns if "FECHA" in c.upper() or "HORA" in c.upper())
    t0=time.perf_counter()
    df[dt]=pd.to_datetime(df[dt], dayfirst=True, errors='coerce')
    df=df.dropna(subset=[dt]).sort_values(dt).reset_index(drop=True)
    df=add_time(df, dt)
    mc=[c for c in ["DD","NO2","PM10","TMP","VV","CO"] if c in df.columns]
    df=make_lags(df, [TARGET_COL]+mc, LAGS).dropna().reset_index(drop=True)
    T['feat']=time.perf_counter()-t0
    cf=[c for c in mc+['hour_sin','hour_cos','month_sin','month_cos'] if c in df.columns]
    t0=time.perf_counter()
    Xc=StandardScaler().fit_transform(df[cf].values)
    best_bic=np.inf
    for k in range(2,6):
        g=GaussianMixture(n_components=k, covariance_type='full', random_state=SEED, n_init=5).fit(Xc)
        best_bic=min(best_bic, g.bic(Xc))
    T['gmm']=time.perf_counter()-t0; T['n']=len(df)
    return T

rows=[]
for station,(folder,fname) in STATIONS.items():
    p = os.path.join(BASE, folder, fname)
    if not os.path.exists(p): print(f"[{station}] MISSING data file"); continue
    print(f"\n[{station}] data file OK"); T = fast_stage(p)
    js = find_json(folder)
    t_models = extract_model_time(js) if js else None
    if js: print(f"  models: {t_models:.1f}s from {os.path.basename(js)}")
    else:  print(f"  [WARN] no metrics JSON found")
    rows.append({'station':station, 'n_clean':T['n'], 't_feat':T['feat'],
                 't_gmm':T['gmm'], 't_models':t_models})

# If any station lacks t_models, estimate from the one with most samples
valid=[r for r in rows if r['t_models'] is not None]
ref = max(valid, key=lambda r: r['n_clean']) if valid else None
for r in rows:
    if r['t_models'] is None and ref:
        r['t_models'] = ref['t_models'] * (r['n_clean']/ref['n_clean'])
        r['estimated'] = True
    r['t_total'] = r['t_feat'] + r['t_gmm'] + (r['t_models'] or 0)

df_out = pd.DataFrame([{
    'Station': r['station'],
    'N_clean': r['n_clean'],
    'Feature eng. (s)': round(r['t_feat'],1),
    'GMM+BIC (s)':      round(r['t_gmm'],1),
    'Model training (s)': None if r['t_models'] is None else round(r['t_models'],1),
    'Total (s)':         round(r['t_total'],1),
    'Total (min)':       round(r['t_total']/60,1),
    'Estimated?':        'Yes' if r.get('estimated') else 'No',
} for r in rows])

print("\n" + "="*100); print("STATION-WISE TRAINING TIMES"); print("="*100)
print(df_out.to_string(index=False))
df_out.to_csv("station_training_times.csv", index=False)
print("\nSaved to station_training_times.csv")


[Asomadilla] data file OK
